# RSNA Knee — Data Audit

Run on **Kaggle** with competition data attached (or locally on CSV metadata only).

Exports summary CSVs you can download into `docs/audit/`.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
import numpy as np

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis",
    "Baker's", "Contusion", "Fracture",
]

CANDIDATES = [
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormalities-detection"),
]
DATA = next((p for p in CANDIDATES if p.exists()), None)
if DATA is None:
    DATA = Path(os.environ.get("RSNA_KNEE_DATA", "data/raw"))

OUT = Path("/kaggle/working/audit") if Path("/kaggle/working").exists() else Path("docs/audit")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA =", DATA)
print("OUT  =", OUT)
assert (DATA / "train.csv").exists(), f"train.csv not found under {DATA}"

In [ ]:
train = pd.read_csv(DATA / "train.csv")
series = pd.read_csv(DATA / "train_series.csv")
print("studies", len(train), "| series", len(series))
print("train columns:", list(train.columns))
train.head(2)

In [ ]:
rows = []
for c in LABEL_COLS:
    if c not in train.columns:
        rows.append({"label": c, "n_non_null": 0, "pct_labeled": 0.0, "n_pos_non_null": 0, "prevalence_among_labeled": np.nan})
        continue
    s = train[c]
    nn = int(s.notna().sum())
    prev = float(s.dropna().mean()) if nn else np.nan
    rows.append({
        "label": c,
        "n_non_null": nn,
        "pct_labeled": float(nn) / len(train),
        "n_pos_non_null": int(s.dropna().sum()) if nn else 0,
        "prevalence_among_labeled": prev,
    })

label_summary = pd.DataFrame(rows)
label_summary.to_csv(OUT / "label_summary.csv", index=False)
label_summary

In [ ]:
plane = series["Anatomical_Plane"].value_counts(dropna=False).rename_axis("plane").reset_index(name="count")
ff = series.groupby(["Fluid_Sensitive", "Fat_Suppression"], dropna=False).size().reset_index(name="count")
per_study = series.groupby("StudyInstanceUID").size().describe()

plane.to_csv(OUT / "plane_counts.csv", index=False)
ff.to_csv(OUT / "fluid_fat_counts.csv", index=False)
pd.DataFrame(per_study).T.to_csv(OUT / "series_per_study_describe.csv", index=False)

print("series per study:\n", per_study)
plane, ff

In [ ]:
reports = train["Report"].fillna("").astype(str)
n_empty = int((reports.str.strip() == "").sum())

def rough_lang(text: str) -> str:
    t = text.lower()
    if not t.strip():
        return "empty"
    if any(ch in t for ch in "àâçéèêëïîôùûüÿœæ"):
        return "likely_fr_or_romance"
    if any("\u3040" <= ch <= "\u30ff" or "\u4e00" <= ch <= "\u9fff" for ch in text):
        return "likely_cjk"
    if any("\u0400" <= ch <= "\u04ff" for ch in text):
        return "likely_cyrillic"
    return "likely_latin"

lang = reports.map(rough_lang).value_counts().rename_axis("bucket").reset_index(name="count")
lang.to_csv(OUT / "report_lang_buckets.csv", index=False)
print("empty reports:", n_empty, f"({n_empty/len(train):.1%})")
lang

In [ ]:
series_root = DATA / "train_series"
spot = {"series_root_exists": series_root.exists(), "n_spot_ok": 0, "n_spot_fail": 0, "transfer_syntaxes": {}}
if series_root.exists():
    try:
        import pydicom
        sample = series.head(8)
        for _, row in sample.iterrows():
            d = series_root / str(row["StudyInstanceUID"]) / str(row["SeriesInstanceUID"])
            dcms = list(d.glob("*.dcm"))[:1] if d.exists() else []
            if not dcms:
                spot["n_spot_fail"] += 1
                continue
            try:
                ds = pydicom.dcmread(str(dcms[0]), stop_before_pixels=True, force=True)
                ts = str(ds.file_meta.get("TransferSyntaxUID", "unknown")) if hasattr(ds, "file_meta") else "unknown"
                spot["transfer_syntaxes"][ts] = spot["transfer_syntaxes"].get(ts, 0) + 1
                spot["n_spot_ok"] += 1
            except Exception as e:
                spot["n_spot_fail"] += 1
                spot.setdefault("errors", []).append(str(e)[:120])
    except ImportError:
        spot["error"] = "pydicom not installed"

(OUT / "dicom_spotcheck.json").write_text(json.dumps(spot, indent=2))
spot

In [ ]:
print("Wrote audit artifacts:")
for p in sorted(OUT.glob("*")):
    print(" -", p, p.stat().st_size, "bytes")
print("\nDownload /kaggle/working/audit/* into the repo as docs/audit/ after Save Version.")